# ================================================================
# BOOTCAMP: Fundamentos de Ingeniería de Datos
# Semana 2: Práctica de CTEs y Window Functions
# ================================================================

**Instructor:** Luciano Argolo  
**Web:** lucianoargolo.com

---

## 🎯 Objetivos

1. Dominar **CTEs (Common Table Expressions)** para organizar queries complejas
2. Aprender **Window Functions** para análisis avanzados
3. Combinar ambas técnicas para resolver problemas reales
4. Practicar con datos reales de NYC Taxi Trips

---

## 📊 Dataset

Usaremos la tabla `samples.nyctaxi.trips` que contiene datos de viajes de taxis en Nueva York.

**Columnas principales:**
- `tpep_pickup_datetime`: Fecha y hora de inicio del viaje
- `tpep_dropoff_datetime`: Fecha y hora de fin del viaje
- `trip_distance`: Distancia del viaje en millas
- `fare_amount`: Tarifa del viaje
- `pickup_zip`: Código postal de inicio
- `dropoff_zip`: Código postal de destino

## Ejercicio 0.1: CTE básica - Estadísticas por zona

**Objetivo:** Usar una CTE para organizar cálculos.

**Pregunta:** Calcula el promedio de tarifa y distancia por zona de inicio (`pickup_zip`). Usa una CTE para calcular primero las estadísticas por zona, y luego muestra solo las zonas con más de 100 viajes.

In [0]:
%sql
-- Ejercicio 0.1: CTE básica - Estadísticas por zona
-- Usamos WITH para crear una CTE que calcula estadísticas por zona
-- Luego filtramos solo las zonas con más de 100 viajes

WITH estadisticas_por_zona AS (
  SELECT 
    pickup_zip,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio
  FROM samples.nyctaxi.trips
  WHERE pickup_zip IS NOT NULL
    AND fare_amount > 0
    AND trip_distance > 0
  GROUP BY pickup_zip
  HAVING cantidad_viajes > 100
)
SELECT 
  pickup_zip,
  cantidad_viajes,
  tarifa_promedio,
  distancia_promedio
FROM estadisticas_por_zona
ORDER BY cantidad_viajes DESC
LIMIT 20;

pickup_zip,cantidad_viajes,tarifa_promedio,distancia_promedio
10001,1227,10.62,2.21
10003,1180,10.99,2.33
10011,1128,10.92,2.29
10021,1017,10.15,2.03
10018,1010,11.42,2.58
10023,1006,10.05,2.12
10028,927,10.18,2.21
10012,831,11.38,2.44
10110,761,10.86,2.31
10065,700,9.77,1.97


**Mismo resultado con subquery:** Más difícil de leer y no podés reutilizar la subconsulta. Con CTEs, si necesitás `estadisticas_por_zona` en otro lado, ya la tenés definida.

In [ ]:
%sql
-- Ejercicio 0.1 (versión con subquery)

SELECT 
  pickup_zip,
  cantidad_viajes,
  tarifa_promedio,
  distancia_promedio
FROM (
  SELECT 
    pickup_zip,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio
  FROM samples.nyctaxi.trips
  WHERE pickup_zip IS NOT NULL
    AND fare_amount > 0
    AND trip_distance > 0
  GROUP BY pickup_zip
  HAVING cantidad_viajes > 100
) estadisticas_por_zona
ORDER BY cantidad_viajes DESC
LIMIT 20;

## Ejercicio 0.2: CTE múltiple - Análisis comparativo

**Objetivo:** Usar múltiples CTEs para organizar cálculos complejos.

**Pregunta:** Compara el promedio de tarifa por hora del día con el promedio general. Usa dos CTEs: una para calcular el promedio por hora, otra para calcular el promedio general, y luego únelas para mostrar la diferencia.

In [0]:
%sql
-- Ejercicio 0.2: CTE múltiple - Análisis comparativo
-- Usamos múltiples CTEs para organizar el cálculo
-- Primero calculamos promedio por hora, luego el promedio general

WITH promedio_por_hora AS (
  SELECT 
    HOUR(tpep_pickup_datetime) AS hora_del_dia,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio_hora
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
  GROUP BY HOUR(tpep_pickup_datetime)
),
promedio_general AS (
  SELECT 
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio_total
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
)
SELECT 
  p.hora_del_dia,
  p.cantidad_viajes,
  p.tarifa_promedio_hora,
  g.tarifa_promedio_total,
  ROUND(p.tarifa_promedio_hora - g.tarifa_promedio_total, 2) AS diferencia,
  ROUND((p.tarifa_promedio_hora - g.tarifa_promedio_total) * 100.0 / g.tarifa_promedio_total, 2) AS porcentaje_diferencia
FROM promedio_por_hora p
CROSS JOIN promedio_general g
ORDER BY p.hora_del_dia;

hora_del_dia,cantidad_viajes,tarifa_promedio_hora,tarifa_promedio_total,diferencia,porcentaje_diferencia
0,744,13.23,12.36,0.87,7.04
1,572,12.77,12.36,0.41,3.32
2,445,13.01,12.36,0.65,5.26
3,307,13.27,12.36,0.91,7.36
4,249,14.79,12.36,2.43,19.66
5,220,14.9,12.36,2.54,20.55
6,475,12.22,12.36,-0.14,-1.13
7,801,11.55,12.36,-0.81,-6.55
8,1035,12.08,12.36,-0.28,-2.27
9,1043,11.99,12.36,-0.37,-2.99


**Mismo resultado con subquery:** Acá se nota más la diferencia. Con CTEs cada paso tiene nombre y se lee de arriba para abajo. Con subqueries tenés que leer de adentro para afuera.

In [ ]:
%sql
-- Ejercicio 0.2 (versión con subqueries)
-- Misma lógica que el CTE, pero anidando las queries
-- Más difícil de leer y debuguear — por eso preferimos CTEs

SELECT 
  p.hora_del_dia,
  p.cantidad_viajes,
  p.tarifa_promedio_hora,
  g.tarifa_promedio_total,
  ROUND(p.tarifa_promedio_hora - g.tarifa_promedio_total, 2) AS diferencia,
  ROUND((p.tarifa_promedio_hora - g.tarifa_promedio_total) * 100.0 / g.tarifa_promedio_total, 2) AS porcentaje_diferencia
FROM (
  SELECT 
    HOUR(tpep_pickup_datetime) AS hora_del_dia,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio_hora
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
  GROUP BY HOUR(tpep_pickup_datetime)
) p
CROSS JOIN (
  SELECT ROUND(AVG(fare_amount), 2) AS tarifa_promedio_total
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
) g
ORDER BY p.hora_del_dia;

# Un ejemplo de Usar Window functions para obtener resultados similares

In [ ]:
%sql
SELECT 
  Distinct(HOUR(tpep_pickup_datetime)) AS hora_del_dia,
  ROUND(AVG(fare_amount) OVER (PARTITION BY HOUR(tpep_pickup_datetime)), 2) AS tarifa_promedio_hora,
  ROUND(AVG(fare_amount) OVER (), 2) AS tarifa_promedio_total,
  ROUND(AVG(fare_amount) OVER (PARTITION BY HOUR(tpep_pickup_datetime)) - AVG(fare_amount) OVER (), 2) AS diferencia,
  ROUND((AVG(fare_amount) OVER (PARTITION BY HOUR(tpep_pickup_datetime)) - AVG(fare_amount) OVER ()) * 100.0 / AVG(fare_amount) OVER (), 2) AS porcentaje_diferencia
FROM samples.nyctaxi.trips
WHERE fare_amount > 0;

## Ejercicio 0.3: CTE para limpieza de datos

**Objetivo:** Usar CTE para filtrar datos antes de analizar.

**Pregunta:** Calcula estadísticas de distancia solo para viajes válidos (distancia > 0, tarifa > 0). Usa una CTE para filtrar primero los datos válidos, y luego calcula mínimo, máximo, promedio y mediana.

In [0]:
%sql
-- Ejercicio 0.3: CTE para limpieza de datos
-- La CTE nos permite filtrar datos inválidos primero
-- Luego trabajamos solo con datos limpios

WITH viajes_validos AS (
  SELECT 
    trip_distance,
    fare_amount
  FROM samples.nyctaxi.trips
  WHERE trip_distance > 0
    AND fare_amount > 0
    AND trip_distance IS NOT NULL
    AND fare_amount IS NOT NULL
)
SELECT 
  COUNT(*) AS total_viajes_validos,
  ROUND(MIN(trip_distance), 2) AS distancia_minima,
  ROUND(MAX(trip_distance), 2) AS distancia_maxima,
  ROUND(AVG(trip_distance), 2) AS distancia_promedio,
  ROUND(PERCENTILE(trip_distance, 0.5), 2) AS distancia_mediana
FROM viajes_validos;

total_viajes_validos,distancia_minima,distancia_maxima,distancia_promedio,distancia_mediana
21847,0.02,30.6,2.86,1.7


## Ejercicio 0.4: Window Function - ROW_NUMBER() para ranking

**Objetivo:** Usar ROW_NUMBER() para crear rankings.

**Pregunta:** Crea un ranking de los viajes más caros. Muestra los top 20 viajes con su fecha, distancia, tarifa y el ranking (1 = más caro).

In [0]:
%sql
-- Ejercicio 0.4: Window Function - ROW_NUMBER() para ranking
-- ROW_NUMBER() asigna números únicos consecutivos (1, 2, 3...)
-- No hay empates, cada fila tiene un número único

SELECT 
  ROW_NUMBER() OVER (ORDER BY fare_amount DESC) AS ranking,
  tpep_pickup_datetime AS fecha,
  ROUND(trip_distance, 2) AS distancia,
  ROUND(fare_amount, 2) AS tarifa
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY fare_amount DESC
LIMIT 20;

ranking,fecha,distancia,tarifa
1,2016-02-12T20:55:19.000Z,20.85,275.0
2,2016-02-29T12:16:16.000Z,0.0,260.0
3,2016-01-30T22:28:42.000Z,0.0,188.0
4,2016-02-17T22:23:14.000Z,25.46,130.0
5,2016-01-28T17:36:17.000Z,21.3,115.0
6,2016-01-04T18:58:23.000Z,0.0,105.0
7,2016-01-16T18:09:15.000Z,0.0,105.0
9,2016-02-22T21:17:27.000Z,30.6,95.0
10,2016-01-04T09:19:53.000Z,5.2,95.0
8,2016-02-24T22:19:55.000Z,12.49,95.0


## Ejercicio 0.5: Window Function - RANK() y DENSE_RANK()

**Objetivo:** Diferenciar entre RANK(), DENSE_RANK() y ROW_NUMBER().

**Pregunta:** Para las 10 zonas con más viajes, crea tres rankings diferentes usando ROW_NUMBER(), RANK() y DENSE_RANK() ordenados por cantidad de viajes. ¿Notas la diferencia?

In [0]:
%sql
-- Ejercicio 0.5: RANK() y DENSE_RANK() con datos reales de NYC Taxi
-- Primero calculamos cantidad de viajes por zona, luego aplicamos las 3 funciones

WITH viajes_por_zona AS (
  SELECT 
    pickup_zip,
    COUNT(*) AS cantidad_viajes
  FROM samples.nyctaxi.trips
  WHERE pickup_zip IS NOT NULL
  GROUP BY pickup_zip
  ORDER BY cantidad_viajes DESC
  LIMIT 10
)
SELECT 
  pickup_zip,
  cantidad_viajes,
  ROW_NUMBER() OVER (ORDER BY cantidad_viajes DESC) AS ranking_row_number,
  RANK()       OVER (ORDER BY cantidad_viajes DESC) AS ranking_rank,
  DENSE_RANK() OVER (ORDER BY cantidad_viajes DESC) AS ranking_dense_rank
FROM viajes_por_zona
ORDER BY cantidad_viajes DESC;

zona,cantidad_viajes,ranking_row_number,ranking_rank,ranking_dense_rank
Zona A,1000,1,1,1
Zona B,850,2,2,2
Zona C,750,3,3,3
Zona D,750,4,3,3
Zona E,750,5,3,3
Zona F,600,6,6,4
Zona G,500,7,7,5
Zona H,500,8,7,5
Zona I,400,9,9,6
Zona J,300,10,10,7


## Ejercicio 0.6: Window Function - Comparar con promedio general

**Objetivo:** Usar AVG() OVER() para comparar con el promedio.

**Pregunta:** Para cada viaje, muestra la tarifa, la distancia, y cómo se compara la tarifa con el promedio general (diferencia y porcentaje de diferencia).

In [0]:
%sql
-- Ejercicio 0.6: Window Function - Comparar con promedio general
-- AVG() OVER() calcula el promedio de TODAS las filas sin agrupar
-- Esto nos permite comparar cada fila individual con el promedio general

SELECT 
  tpep_pickup_datetime AS fecha,
  ROUND(fare_amount, 2) AS tarifa,
  ROUND(trip_distance, 2) AS distancia,
  ROUND(AVG(fare_amount) OVER(), 2) AS tarifa_promedio_general,
  ROUND(fare_amount - AVG(fare_amount) OVER(), 2) AS diferencia_con_promedio,
  ROUND((fare_amount - AVG(fare_amount) OVER()) * 100.0 / AVG(fare_amount) OVER(), 2) AS porcentaje_diferencia
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY fecha DESC
LIMIT 20;

fecha,tarifa,distancia,tarifa_promedio_general,diferencia_con_promedio,porcentaje_diferencia
2016-02-29T23:51:20.000Z,6.5,0.91,9.85,-5.86,-47.39
2016-02-29T23:51:06.000Z,7.5,1.4,11.9,-4.86,-39.3
2016-02-29T23:50:27.000Z,6.0,1.35,11.44,-6.36,-51.44
2016-02-29T23:39:15.000Z,12.5,2.8,10.98,0.14,1.17
2016-02-29T23:33:22.000Z,5.5,0.8,10.62,-6.86,-55.49
2016-02-29T23:32:16.000Z,25.5,8.72,30.64,13.14,106.39
2016-02-29T23:29:04.000Z,6.0,1.0,10.98,-6.36,-51.44
2016-02-29T23:23:23.000Z,15.5,3.5,10.62,3.14,25.45
2016-02-29T23:23:16.000Z,4.0,0.51,12.03,-8.36,-67.63
2016-02-29T23:21:38.000Z,11.0,2.33,12.3,-1.36,-10.97


## Ejercicio 0.7: Window Function - Promedio por partición (PARTITION BY)

**Objetivo:** Usar PARTITION BY para calcular promedios por grupo.

**Pregunta:** Para cada viaje, muestra: fecha, tarifa, zona de inicio (pickup_zip), y el promedio de tarifa para esa misma zona (ej: si es zona 10001, muestra el promedio de todos los viajes de la zona 10001).

**Diferencia clave con 0.6:** En 0.6 usamos `AVG() OVER()` sin PARTITION BY → promedio **general**. Acá usamos `PARTITION BY pickup_zip` → promedio **por zona**.

In [ ]:
%sql
-- Ejercicio 0.7: PARTITION BY para calcular promedios por zona
-- AVG() OVER (PARTITION BY pickup_zip) calcula el promedio
-- solo dentro del grupo de viajes de la misma zona

SELECT 
  tpep_pickup_datetime AS fecha,
  ROUND(fare_amount, 2) AS tarifa,
  pickup_zip,
  ROUND(AVG(fare_amount) OVER (PARTITION BY pickup_zip), 2) AS tarifa_promedio_zona,
  ROUND(fare_amount - AVG(fare_amount) OVER (PARTITION BY pickup_zip), 2) AS diferencia_con_zona
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
  AND pickup_zip IS NOT NULL
ORDER BY pickup_zip, fare_amount DESC
LIMIT 50;

## Ejercicio 0.8: Window Function - LAG() para comparar con anterior

**Objetivo:** Usar LAG() para comparar con el registro anterior.

**Pregunta:** Ordena los viajes por fecha y muestra: fecha, tarifa, y la tarifa del viaje anterior (si existe). Esto te permite ver si hay tendencias temporales.

In [0]:
%sql
-- Ejercicio 0.8: Window Function - LAG() para comparar con anterior
-- LAG() obtiene el valor de la fila anterior según el orden especificado
-- Útil para comparar valores consecutivos y detectar tendencias

SELECT 
  tpep_pickup_datetime AS fecha,
  ROUND(fare_amount, 2) AS tarifa_actual,
  LAG(fare_amount) OVER (ORDER BY tpep_pickup_datetime) AS tarifa_anterior,
  ROUND(fare_amount - LAG(fare_amount) OVER (ORDER BY tpep_pickup_datetime), 2) AS diferencia_con_anterior
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY tpep_pickup_datetime
LIMIT 50;

fecha,tarifa_actual,tarifa_anterior,diferencia_con_anterior
2016-01-01T00:04:30.000Z,5.0,null,null
2016-01-01T00:11:29.000Z,24.5,5.0,19.5
2016-01-01T00:12:48.000Z,5.0,24.5,-19.5
2016-01-01T00:13:37.000Z,10.0,5.0,5.0
2016-01-01T00:15:46.000Z,8.5,10.0,-1.5
2016-01-01T00:15:57.000Z,11.0,8.5,2.5
2016-01-01T00:20:57.000Z,5.5,11.0,-5.5
2016-01-01T00:22:30.000Z,4.0,5.5,-1.5
2016-01-01T00:23:34.000Z,13.0,4.0,9.0
2016-01-01T00:24:29.000Z,10.0,13.0,-3.0


## Ejercicio 0.9: Window Function - SUM() acumulado

**Objetivo:** Usar SUM() OVER() para calcular acumulados.

**Pregunta:** Ordena los viajes por fecha y muestra: fecha, tarifa, y el total acumulado de tarifas hasta esa fecha (running total).

In [0]:
%sql
-- Ejercicio 0.9: Window Function - SUM() acumulado
-- SUM() OVER() con ORDER BY calcula un total acumulado (running total)
-- ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW suma desde el inicio hasta la fila actual

SELECT 
  tpep_pickup_datetime AS fecha,
  ROUND(fare_amount, 2) AS tarifa,
  ROUND(SUM(fare_amount) OVER (
    ORDER BY tpep_pickup_datetime 
  ), 2) AS total_acumulado
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY tpep_pickup_datetime
LIMIT 100;

fecha,tarifa,total_acumulado
2016-01-01T00:04:30.000Z,5.0,5.0
2016-01-01T00:11:29.000Z,24.5,29.5
2016-01-01T00:12:48.000Z,5.0,34.5
2016-01-01T00:13:37.000Z,10.0,44.5
2016-01-01T00:15:46.000Z,8.5,53.0
2016-01-01T00:15:57.000Z,11.0,64.0
2016-01-01T00:20:57.000Z,5.5,69.5
2016-01-01T00:22:30.000Z,4.0,73.5
2016-01-01T00:23:34.000Z,13.0,86.5
2016-01-01T00:24:29.000Z,10.0,96.5


## Ejercicio 0.10: Combinando CTEs y Window Functions

**Objetivo:** Combinar ambas técnicas en una query compleja.

**Pregunta:** Usa CTEs para:
1. Filtrar viajes válidos (distancia > 0, tarifa > 0)
2. Calcular estadísticas por zona (promedio, máximo, mínimo)
3. Luego usa Window Functions para rankear las zonas por promedio de tarifa
4. Muestra el top 10 zonas más caras con su ranking

In [0]:
%sql
-- Ejercicio 0.10: Combinando CTEs y Window Functions
-- Este ejercicio demuestra cómo combinar ambas técnicas
-- CTEs para organizar y limpiar datos, Window Functions para análisis avanzado

WITH viajes_validos AS (
  -- Paso 1: Filtrar viajes válidos
  SELECT 
    pickup_zip,
    fare_amount,
    trip_distance
  FROM samples.nyctaxi.trips
  WHERE pickup_zip IS NOT NULL
    AND fare_amount > 0
    AND trip_distance > 0
),
estadisticas_por_zona AS (
  -- Paso 2: Calcular estadísticas por zona
  SELECT 
    pickup_zip,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio,
    ROUND(MAX(fare_amount), 2) AS tarifa_maxima,
    ROUND(MIN(fare_amount), 2) AS tarifa_minima
  FROM viajes_validos
  GROUP BY pickup_zip
  HAVING COUNT(*) >= 50  -- Solo zonas con al menos 50 viajes
)
SELECT 
  -- Paso 3: Usar Window Function para rankear
  ROW_NUMBER() OVER (ORDER BY tarifa_promedio DESC) AS ranking,
  pickup_zip,
  cantidad_viajes,
  tarifa_promedio,
  tarifa_maxima,
  tarifa_minima
FROM estadisticas_por_zona
ORDER BY tarifa_promedio DESC
LIMIT 10;

ranking,pickup_zip,cantidad_viajes,tarifa_promedio,tarifa_maxima,tarifa_minima
1,11422,421,45.23,82.5,3.0
2,11371,482,30.63,95.0,4.5
3,10280,103,16.33,55.0,3.5
4,10005,101,15.87,52.0,3.5
5,10271,124,15.52,41.5,3.5
6,10006,124,15.32,43.0,3.0
7,10282,163,14.46,85.0,2.5
8,10007,167,13.89,56.5,2.5
9,10013,273,13.84,275.0,3.0
10,11201,79,13.63,52.0,3.0


---

## 💡 Resumen de Conceptos Clave

### CTEs (Common Table Expressions)
- **Cuándo usarlas:** Para organizar queries complejas, reutilizar subconsultas, mejorar legibilidad
- **Sintaxis:** `WITH nombre_cte AS (SELECT ...)`
- **Ventajas:** Código más limpio, fácil de mantener, permite múltiples CTEs en una query

### Window Functions
- **ROW_NUMBER():** Asigna números únicos consecutivos (1, 2, 3...)
- **RANK():** Ranking con saltos en empates (1, 2, 2, 4...)
- **DENSE_RANK():** Ranking sin saltos en empates (1, 2, 2, 3...)
- **AVG() OVER():** Calcula promedio sin agrupar filas
- **SUM() OVER():** Calcula suma acumulada
- **LAG():** Obtiene valor de la fila anterior
- **PARTITION BY:** Agrupa filas antes de aplicar la función

### Combinando CTEs y Window Functions
- Usa CTEs para limpiar y preparar datos
- Usa Window Functions para análisis comparativos y rankings
- Ambas técnicas se complementan perfectamente

---

**¡Ahora estás listo para aplicar estas técnicas en el EDA de propiedades! 🚀**